# 06c — Comparing to PCA

## The One-Sentence Version
When does the neural network overhead actually pay off?

## What You'll Build Intuition For
- How PCA, autoencoders, and VAEs compare on the same data
- Where autoencoders genuinely beat PCA (nonlinear structure)
- Where PCA is sufficient (linear data, small datasets)
- The computational cost trade-off

## Prerequisites
- [06a — Autoencoder Basics](06a_autoencoder_basics.ipynb)
- [06b — Variational Autoencoders](06b_variational_autoencoders.ipynb)
- [03c — PCA Applied](../03_linear_methods/03c_pca_applied.ipynb)

## The Story

We now have three compression methods:
- **PCA**: linear, fast, deterministic, no training needed
- **Autoencoder**: nonlinear, requires training, captures curved structure
- **VAE**: nonlinear, smooth latent space, can generate new data

Each adds complexity. Is it worth it? The honest answer: *it depends on your data*. If the structure is linear, PCA wins — it's faster, simpler, and just as accurate. If the structure is nonlinear, autoencoders can dramatically outperform PCA.

This notebook is about building the judgement to know which tool to reach for.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data, make_swiss_roll_data

apply_style()
rng = np.random.default_rng(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Redefine models from 06a and 06b
class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        return self.encoder(x)

class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU())
        self.mu = nn.Linear(128, latent_dim)
        self.logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, input_dim), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        return self.decoder(z), mu, logvar

def train_ae(X_tensor, latent_dim, epochs=100):
    model = SimpleAutoencoder(X_tensor.shape[1], latent_dim).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)
    for _ in range(epochs):
        for (batch,) in loader:
            loss = nn.functional.mse_loss(model(batch), batch)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    return model

## Same Task, Three Methods

Compress digits (64D) to 2D using PCA, autoencoder, and VAE. Compare latent spaces side by side.

In [ ]:
data, target, images = make_digit_data()
X_norm = data / 16.0
X_tensor = torch.FloatTensor(X_norm).to(device)

# PCA
Z_pca = PCA(n_components=2).fit_transform(X_norm)

# Autoencoder
ae = train_ae(X_tensor, latent_dim=2, epochs=150)
ae.eval()
with torch.no_grad():
    Z_ae = ae.encode(X_tensor).cpu().numpy()

# VAE
vae = VAE(64, 2).to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_tensor), batch_size=64, shuffle=True)
for _ in range(200):
    for (batch,) in loader:
        recon, mu, logvar = vae(batch)
        rl = nn.functional.binary_cross_entropy(recon, batch, reduction='sum')
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = rl + kl
        opt.zero_grad()
        loss.backward()
        opt.step()

vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_tensor)
    Z_vae = mu_all.cpu().numpy()

# Plot side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
methods = [("PCA", Z_pca), ("Autoencoder", Z_ae), ("VAE", Z_vae)]

for ax, (name, Z) in zip(axes, methods):
    for digit in range(10):
        mask = target == digit
        ax.scatter(Z[mask, 0], Z[mask, 1], s=8, alpha=0.5,
                   color=PALETTE[digit % len(PALETTE)], label=str(digit))
    ax.set_title(name, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

axes[-1].legend(title="Digit", markerscale=3, fontsize=7, ncol=2, loc="upper right")
fig.suptitle("Digits → 2D: PCA vs Autoencoder vs VAE", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06c_three_methods.png", dpi=150, bbox_inches="tight")
plt.show()

## Reconstruction Error Curves

How does reconstruction quality compare as we increase the number of latent dimensions?

In [ ]:
latent_dims = [2, 5, 10, 20, 30]
pca_mses = []
ae_mses = []

for ld in latent_dims:
    # PCA
    pca = PCA(n_components=ld)
    Z = pca.fit_transform(X_norm)
    X_pca_recon = pca.inverse_transform(Z)
    pca_mses.append(np.mean((X_norm - X_pca_recon) ** 2))

    # Autoencoder
    model = train_ae(X_tensor, latent_dim=ld, epochs=100)
    model.eval()
    with torch.no_grad():
        recon = model(X_tensor).cpu().numpy()
    ae_mses.append(np.mean((X_norm - recon) ** 2))

    print(f"dim={ld:>2}: PCA MSE={pca_mses[-1]:.4f}, AE MSE={ae_mses[-1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(latent_dims, pca_mses, "s--", color=COLOURS["accent"], linewidth=2,
        markersize=8, label="PCA")
ax.plot(latent_dims, ae_mses, "o-", color=COLOURS["signal"], linewidth=2,
        markersize=8, label="Autoencoder")
ax.set_xlabel("Latent Dimensions")
ax.set_ylabel("Reconstruction MSE")
ax.set_title("Reconstruction error: autoencoder advantage at low dimensions")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/06c_recon_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nThe autoencoder's nonlinear compression is most useful at very low dimensions.")
print("As dimensions increase, PCA catches up — the remaining structure is mostly linear.")

## The Nonlinear Advantage

Where autoencoders truly shine: data with curved manifold structure that PCA can't capture.

In [ ]:
# Swiss roll: curved manifold in 3D → compress to 2D
X_swiss, colour_swiss = make_swiss_roll_data(n=1500, noise=0.3, seed=42)
X_swiss_norm = (X_swiss - X_swiss.mean(axis=0)) / X_swiss.std(axis=0)
X_swiss_t = torch.FloatTensor(X_swiss_norm).to(device)

# PCA
Z_pca_swiss = PCA(n_components=2).fit_transform(X_swiss_norm)

# Autoencoder
ae_swiss = train_ae(X_swiss_t, latent_dim=2, epochs=200)
ae_swiss.eval()
with torch.no_grad():
    Z_ae_swiss = ae_swiss.encode(X_swiss_t).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(Z_pca_swiss[:, 0], Z_pca_swiss[:, 1], s=5, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax1.set_title("PCA: can't unroll the manifold")
ax1.set_xticks([])
ax1.set_yticks([])

ax2.scatter(Z_ae_swiss[:, 0], Z_ae_swiss[:, 1], s=5, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax2.set_title("Autoencoder: learns to unroll")
ax2.set_xticks([])
ax2.set_yticks([])

fig.suptitle("Swiss Roll: the autoencoder's nonlinear advantage", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06c_swiss_roll.png", dpi=150, bbox_inches="tight")
plt.show()

print("PCA projects through the layers — colours mix.")
print("The autoencoder can learn the curved unrolling that PCA can't.")
print("\nBut: the autoencoder needs training data. PCA doesn't.")

## Computational Cost

Autoencoders are powerful but expensive. Let's measure the gap.

In [ ]:
# Time PCA vs autoencoder on digits
X_tensor_full = torch.FloatTensor(X_norm).to(device)

# PCA timing
start = time.time()
pca = PCA(n_components=10)
pca.fit_transform(X_norm)
pca_time = time.time() - start

# Autoencoder timing (including training)
start = time.time()
_ = train_ae(X_tensor_full, latent_dim=10, epochs=100)
ae_time = time.time() - start

fig, ax = plt.subplots(figsize=(8, 5))
methods = ["PCA", "Autoencoder\n(100 epochs)"]
times = [pca_time, ae_time]
colours_bar = [COLOURS["signal"], COLOURS["noise"]]

ax.bar(methods, times, color=colours_bar, alpha=0.8, edgecolor="white")
for i, t in enumerate(times):
    ax.text(i, t + max(times) * 0.03, f"{t:.2f}s", ha="center", fontsize=13,
            fontweight="bold")
ax.set_ylabel("Time (seconds)")
ax.set_title(f"Computational cost: PCA vs Autoencoder (digits, n={len(X_norm)})")
plt.tight_layout()
plt.savefig("visuals/06c_timing.png", dpi=150, bbox_inches="tight")
plt.show()

speedup = ae_time / pca_time if pca_time > 0 else float('inf')
print(f"PCA: {pca_time:.3f}s")
print(f"Autoencoder: {ae_time:.2f}s")
print(f"PCA is ~{speedup:.0f}× faster.")
print(f"\nThe gap grows with dataset size and network complexity.")

## When to Use What

| Method | Strengths | Weaknesses | Use When |
|--------|-----------|------------|----------|
| **PCA** | Fast, deterministic, no training | Linear only | First pass on any data; sufficient for linear structure |
| **Autoencoder** | Captures nonlinear structure | Needs training, non-deterministic | Data has curved manifold structure PCA misses |
| **VAE** | Smooth latent space, can generate | More complex, may under-reconstruct | Need to interpolate, sample, or generate new data |

> **Rule of thumb:** Always try PCA first. Only go neural if PCA clearly fails — the reconstruction error is high, or the latent space shows obvious overlap between classes that should be separate.

<details>
<summary><b>The Maths Behind This</b></summary>

**Why PCA is optimal for linear data:** PCA finds the rank-$k$ approximation that minimizes $\|X - X_k\|_F^2$ (Eckart-Young theorem). If the data truly lives near a $k$-dimensional linear subspace, no method — linear or nonlinear — can do better.

**Why autoencoders can beat PCA on nonlinear data:** PCA is constrained to affine subspaces. An autoencoder with nonlinear activations can approximate any continuous mapping $f: \mathbb{R}^D \to \mathbb{R}^d$ (universal approximation). When the data manifold is curved, this extra capacity reduces reconstruction error.

**The cost of capacity:** More capacity means more parameters, more training time, and risk of overfitting. The autoencoder's advantage is largest when:
1. The data manifold is strongly nonlinear
2. There is enough training data to learn the manifold
3. The bottleneck is very narrow (few latent dims)

When any of these conditions fails, PCA is the better choice.

</details>

## Where This Matters

**Model selection in practice:** A data scientist choosing between PCA and an autoencoder for a new dataset should follow this workflow: (1) try PCA, (2) check explained variance and reconstruction, (3) only invest in neural methods if PCA clearly fails. This saves days of unnecessary GPU time.

**Resource-constrained environments:** In deployed systems (edge devices, real-time pipelines), PCA's millisecond inference time can be critical. Autoencoders may provide marginal quality gains that don't justify the computational cost.

## Feynman Check

1. **On what type of data does PCA match or beat autoencoders?**

2. **What are three practical reasons to prefer PCA over an autoencoder?**

3. **When is the extra complexity of a VAE justified?**

---

Module 06 is complete. You now understand learned compression — from simple autoencoders to VAEs — and crucially, when they're worth the overhead.

In **Module 07: Applied Thinking**, we'll pull everything together: decision frameworks, simulation parameter spaces, and the philosophy of compression.